In [1]:
from scapy.all import rdpcap

packets = rdpcap("deneme.pcapng")

print("Packet Count:", len(packets))

print("\nİlk paket:")
print(packets[0].summary())

Packet Count: 51

İlk paket:
Ether / IP / UDP / DNS Qry b'login.microsoftonline.com.'


In [2]:
from collections import defaultdict
from scapy.all import rdpcap, IP, TCP, UDP

packets = rdpcap("deneme.pcapng")

flows = defaultdict(int)

for pkt in packets:
    if IP not in pkt:
        continue

    proto = pkt[IP].proto

    if TCP in pkt:
        sport = pkt[TCP].sport
        dport = pkt[TCP].dport

    elif UDP in pkt:
        sport = pkt[UDP].sport
        dport = pkt[UDP].dport

    else:
        sport = 0
        dport = 0

    key = (
        pkt[IP].src,
        pkt[IP].dst,
        sport,
        dport,
        proto
    )

    flows[key] += 1

print("Flow sayisi:", len(flows))

for flow, cnt in list(flows.items())[:10]:
    print(cnt, flow)

Flow sayisi: 10
1 ('10.34.2.36', '10.36.0.30', 49198, 53, 17)
1 ('10.36.0.30', '10.34.2.36', 53, 49198, 17)
11 ('10.34.2.36', '20.190.177.19', 50235, 443, 6)
23 ('20.190.177.19', '10.34.2.36', 443, 50235, 6)
5 ('10.34.10.60', '10.34.2.36', 63499, 7680, 6)
3 ('10.34.2.36', '10.34.10.60', 7680, 63499, 6)
1 ('10.5.28.49', '10.34.2.36', 7680, 54129, 6)
1 ('10.34.2.36', '10.5.28.49', 54129, 7680, 6)
1 ('10.34.2.36', '34.102.128.190', 63749, 443, 6)
1 ('10.34.2.36', '10.34.0.31', 50236, 135, 6)


In [3]:
from collections import defaultdict
from scapy.all import rdpcap, IP, TCP, UDP

packets = rdpcap("deneme.pcapng")

flows = {}

for pkt in packets:

    if IP not in pkt:
        continue

    proto = pkt[IP].proto

    if TCP in pkt:
        sport = pkt[TCP].sport
        dport = pkt[TCP].dport

    elif UDP in pkt:
        sport = pkt[UDP].sport
        dport = pkt[UDP].dport

    else:
        sport = 0
        dport = 0

    key = (
        pkt[IP].src,
        pkt[IP].dst,
        sport,
        dport,
        proto
    )

    if key not in flows:

        flows[key] = {
            "first": pkt.time,
            "last": pkt.time,
            "bytes": 0,
            "pkts": 0
        }

    flows[key]["last"] = pkt.time
    flows[key]["bytes"] += len(pkt)
    flows[key]["pkts"] += 1


for flow, data in flows.items():

    duration = (
        data["last"] -
        data["first"]
    ) * 1000

    print(
        "\nFLOW:",
        flow
    )

    print(
        "PKTS:",
        data["pkts"]
    )

    print(
        "BYTES:",
        data["bytes"]
    )

    print(
        "DURATION_MS:",
        round(duration,2)
    )


FLOW: ('10.34.2.36', '10.36.0.30', 49198, 53, 17)
PKTS: 1
BYTES: 85
DURATION_MS: 0.00

FLOW: ('10.36.0.30', '10.34.2.36', 53, 49198, 17)
PKTS: 1
BYTES: 319
DURATION_MS: 0.00

FLOW: ('10.34.2.36', '20.190.177.19', 50235, 443, 6)
PKTS: 11
BYTES: 5156
DURATION_MS: 396.46

FLOW: ('20.190.177.19', '10.34.2.36', 443, 50235, 6)
PKTS: 23
BYTES: 28393
DURATION_MS: 217.62

FLOW: ('10.34.10.60', '10.34.2.36', 63499, 7680, 6)
PKTS: 5
BYTES: 375
DURATION_MS: 10.96

FLOW: ('10.34.2.36', '10.34.10.60', 7680, 63499, 6)
PKTS: 3
BYTES: 174
DURATION_MS: 10.90

FLOW: ('10.5.28.49', '10.34.2.36', 7680, 54129, 6)
PKTS: 1
BYTES: 60
DURATION_MS: 0.00

FLOW: ('10.34.2.36', '10.5.28.49', 54129, 7680, 6)
PKTS: 1
BYTES: 54
DURATION_MS: 0.00

FLOW: ('10.34.2.36', '34.102.128.190', 63749, 443, 6)
PKTS: 1
BYTES: 54
DURATION_MS: 0.00

FLOW: ('10.34.2.36', '10.34.0.31', 50236, 135, 6)
PKTS: 1
BYTES: 66
DURATION_MS: 0.00


In [4]:
from collections import defaultdict
from scapy.all import rdpcap, IP, TCP, UDP

packets = rdpcap("deneme.pcapng")

flows = {}

for pkt in packets:

    if IP not in pkt:
        continue

    proto = pkt[IP].proto

    if TCP in pkt:
        sport = pkt[TCP].sport
        dport = pkt[TCP].dport

    elif UDP in pkt:
        sport = pkt[UDP].sport
        dport = pkt[UDP].dport

    else:
        sport = 0
        dport = 0

    key = (
        pkt[IP].src,
        pkt[IP].dst,
        sport,
        dport,
        proto
    )

    if key not in flows:

        flows[key] = {
            "first_ts": float(pkt.time),
            "last_ts": float(pkt.time),

            "bytes": 0,
            "pkts": 0,

            "min_pkt": 999999,
            "max_pkt": 0,

            "ttl_min": 999999,
            "ttl_max": 0,

            "tcp_flags": 0,
        }

    length = len(pkt)

    flows[key]["bytes"] += length
    flows[key]["pkts"] += 1

    flows[key]["last_ts"] = float(pkt.time)

    flows[key]["min_pkt"] = min(
        flows[key]["min_pkt"],
        length
    )

    flows[key]["max_pkt"] = max(
        flows[key]["max_pkt"],
        length
    )

    flows[key]["ttl_min"] = min(
        flows[key]["ttl_min"],
        pkt[IP].ttl
    )

    flows[key]["ttl_max"] = max(
        flows[key]["ttl_max"],
        pkt[IP].ttl
    )

    if TCP in pkt:
        flows[key]["tcp_flags"] |= int(pkt[TCP].flags)

In [5]:
import pandas as pd

rows = []

for flow, stat in flows.items():

    duration_ms = (
        stat["last_ts"] -
        stat["first_ts"]
    ) * 1000

    row = {
        "FLOW": flow,

        "PROTOCOL": flow[4],

        "IN_BYTES": stat["bytes"],
        "IN_PKTS": stat["pkts"],

        "FLOW_DURATION_MILLISECONDS": duration_ms,

        "MIN_TTL": stat["ttl_min"],
        "MAX_TTL": stat["ttl_max"],

        "LONGEST_FLOW_PKT": stat["max_pkt"],
        "SHORTEST_FLOW_PKT": stat["min_pkt"],

        "MIN_IP_PKT_LEN": stat["min_pkt"],
        "MAX_IP_PKT_LEN": stat["max_pkt"],

        "TCP_FLAGS": stat["tcp_flags"]
    }

    rows.append(row)

df_features = pd.DataFrame(rows)

print(df_features.head())

                                         FLOW  PROTOCOL  IN_BYTES  IN_PKTS  \
0     (10.34.2.36, 10.36.0.30, 49198, 53, 17)        17        85        1   
1     (10.36.0.30, 10.34.2.36, 53, 49198, 17)        17       319        1   
2  (10.34.2.36, 20.190.177.19, 50235, 443, 6)         6      5156       11   
3  (20.190.177.19, 10.34.2.36, 443, 50235, 6)         6     28393       23   
4   (10.34.10.60, 10.34.2.36, 63499, 7680, 6)         6       375        5   

   FLOW_DURATION_MILLISECONDS  MIN_TTL  MAX_TTL  LONGEST_FLOW_PKT  \
0                    0.000000      128      128                85   
1                    0.000000      123      123               319   
2                  396.456003      128      128              3310   
3                  217.619658      107      107              1514   
4                   10.958672      127      127               129   

   SHORTEST_FLOW_PKT  MIN_IP_PKT_LEN  MAX_IP_PKT_LEN  TCP_FLAGS  
0                 85              85              

In [7]:
from scapy.all import rdpcap, IP, TCP, UDP
import pandas as pd

packets = rdpcap("deneme.pcapng")

flows = {}

for pkt in packets:

    if IP not in pkt:
        continue

    proto = pkt[IP].proto

    if TCP in pkt:
        sport = pkt[TCP].sport
        dport = pkt[TCP].dport

    elif UDP in pkt:
        sport = pkt[UDP].sport
        dport = pkt[UDP].dport

    else:
        sport = 0
        dport = 0

    src = pkt[IP].src
    dst = pkt[IP].dst

    # ---------------------------------------------------
    # Bidirectional canonical key
    # ---------------------------------------------------

    if (src, sport) < (dst, dport):

        canonical_key = (
            src,
            dst,
            sport,
            dport,
            proto
        )

        direction = "forward"

    else:

        canonical_key = (
            dst,
            src,
            dport,
            sport,
            proto
        )

        direction = "backward"

    # ---------------------------------------------------

    if canonical_key not in flows:

        flows[canonical_key] = {

            "first_ts": float(pkt.time),
            "last_ts": float(pkt.time),

            "IN_BYTES": 0,
            "OUT_BYTES": 0,

            "IN_PKTS": 0,
            "OUT_PKTS": 0,

            "MIN_PKT": 999999,
            "MAX_PKT": 0,

            "MIN_TTL": 255,
            "MAX_TTL": 0,

            "TCP_FLAGS": 0,
        }

    flow = flows[canonical_key]

    pkt_len = len(pkt)

    flow["last_ts"] = float(pkt.time)

    if direction == "forward":

        flow["IN_PKTS"] += 1
        flow["IN_BYTES"] += pkt_len

    else:

        flow["OUT_PKTS"] += 1
        flow["OUT_BYTES"] += pkt_len

    flow["MIN_PKT"] = min(
        flow["MIN_PKT"],
        pkt_len
    )

    flow["MAX_PKT"] = max(
        flow["MAX_PKT"],
        pkt_len
    )

    flow["MIN_TTL"] = min(
        flow["MIN_TTL"],
        pkt[IP].ttl
    )

    flow["MAX_TTL"] = max(
        flow["MAX_TTL"],
        pkt[IP].ttl
    )

    if TCP in pkt:
        flow["TCP_FLAGS"] |= int(pkt[TCP].flags)

# ==================================================
# DataFrame
# ==================================================

rows = []

for flow_key, flow in flows.items():

    duration_ms = (
        flow["last_ts"]
        - flow["first_ts"]
    ) * 1000

    row = {

        "FLOW": flow_key,

        "PROTOCOL": flow_key[4],

        "IN_BYTES": flow["IN_BYTES"],
        "OUT_BYTES": flow["OUT_BYTES"],

        "IN_PKTS": flow["IN_PKTS"],
        "OUT_PKTS": flow["OUT_PKTS"],

        "FLOW_DURATION_MILLISECONDS":
            duration_ms,

        "MIN_TTL": flow["MIN_TTL"],
        "MAX_TTL": flow["MAX_TTL"],

        "LONGEST_FLOW_PKT":
            flow["MAX_PKT"],

        "SHORTEST_FLOW_PKT":
            flow["MIN_PKT"],

        "MIN_IP_PKT_LEN":
            flow["MIN_PKT"],

        "MAX_IP_PKT_LEN":
            flow["MAX_PKT"],

        "TCP_FLAGS":
            flow["TCP_FLAGS"]
    }

    rows.append(row)

df_features = pd.DataFrame(rows)

print(df_features)

                                          FLOW  PROTOCOL  IN_BYTES  OUT_BYTES  \
0      (10.34.2.36, 10.36.0.30, 49198, 53, 17)        17        85        319   
1   (10.34.2.36, 20.190.177.19, 50235, 443, 6)         6      5156      28393   
2    (10.34.10.60, 10.34.2.36, 63499, 7680, 6)         6       375        174   
3     (10.34.2.36, 10.5.28.49, 54129, 7680, 6)         6        54         60   
4  (10.34.2.36, 34.102.128.190, 63749, 443, 6)         6        54          0   
5      (10.34.0.31, 10.34.2.36, 135, 50236, 6)         6         0         66   

   IN_PKTS  OUT_PKTS  FLOW_DURATION_MILLISECONDS  MIN_TTL  MAX_TTL  \
0        1         1                    3.135443      123      128   
1       11        23                  396.456003      107      128   
2        5         3                   11.039734      127      128   
3        1         1                   41.613340      121      128   
4        1         0                    0.000000      128      128   
5        0  